# Episode 3 — LEDs, switches and push buttons

The LEDs, slide switches and push buttons are wired to **AXI GPIO** blocks in the PL.
An AXI GPIO block is a register: writing it drives output pins, reading it returns input pins.

| Python object | Block | Pins | Direction |
| --- | --- | --- | --- |
| `base.leds` | `leds_gpio` | 4 LEDs | output |
| `base.switches` | `switches_gpio` | 4 slide switches | input |
| `base.buttons` | `btns_gpio` | 2 push buttons | input |

You can use one pin (`base.leds[2]`) or a slice of pins (`base.leds[0:4]`), like a Python list.

In [ ]:
from pynq.overlays.base import BaseOverlay
import time

base = BaseOverlay("base.bit")

## 1. One LED at a time

In [ ]:
base.leds[0].on()
time.sleep(0.5)
base.leds[0].off()
base.leds[1].toggle()          # toggle flips the current state
time.sleep(0.5)
base.leds[1].toggle()

## 2. All four LEDs at once

`write()` on a slice takes a number: bit 0 is LED 0, bit 3 is LED 3.

In [ ]:
for pattern in [0b1010, 0b0101, 0b1111, 0b0000]:
    base.leds[0:4].write(pattern)
    time.sleep(0.4)

## 3. A running light

In [ ]:
order = [0, 1, 2, 3, 2, 1]
for step in range(3 * len(order)):
    base.leds[0:4].write(1 << order[step % len(order)])
    time.sleep(0.1)
base.leds[0:4].off()

## 4. A binary counter

In [ ]:
for n in range(16):
    base.leds[0:4].write(n)
    time.sleep(0.25)
base.leds[0:4].off()

## 5. Reading the slide switches

Move some switches and run the cell again.

In [ ]:
value = base.switches.read()
print("All switches: {:04b}".format(value))
for i in range(4):
    print("  switch", i, "is", "ON" if base.switches[i].read() else "off")

## 6. Switches drive the LEDs

Python reads the switches and writes the LEDs 20 times a second, for 10 seconds.
This loop is called **polling**.

In [ ]:
end = time.time() + 10
while time.time() < end:
    base.leds[0:4].write(base.switches.read())
    time.sleep(0.05)
base.leds[0:4].off()

## 7. Counting button presses

A press is the moment the button value changes from 0 to 1 (a **rising edge**).
Real buttons bounce for a few milliseconds, so after each press we wait for the
release before counting again. The count is shown on the LEDs in binary.

Press button 0 as often as you like in 10 seconds.

In [ ]:
count = 0
last = 0
end = time.time() + 10
while time.time() < end:
    now = base.buttons[0].read()
    if now == 1 and last == 0:          # rising edge = a new press
        count += 1
        base.leds[0:4].write(count & 0xF)
        time.sleep(0.02)                # ignore the bounce
    last = now
    time.sleep(0.005)
print("Button 0 was pressed", count, "times")

## Try it yourself

Change the running light so that button 1 reverses its direction while it runs.

In [ ]:
# your code here